# Assemble Word Counts + Firing Rates

This notebook builds word-aligned neural features for one patient from the `vad_new` layout.

It uses:
- `{patient}_transcripts.csv` from `speech_extraction/assemble_transcripts.ipynb`
- interval-level `binary_spiketrain.npy` files from `signal_processing/spike_thresholding.ipynb`

Outputs are saved row-aligned with the transcript CSV under `vad_new/{patient}/neural_embeddings/`:
- `word_counts.npy` -> `(n_words, n_channels)`
- `word_spike_counts_offsets_all.npy` -> `(1, n_words, n_channels)` for offset `0`
- `word_frs.npy` -> `(n_words, n_channels)`
- `word_durs.npy` -> `(n_words,)`
- `word_neural_manifest.json`

A lightweight transcript copy with neural metadata is also written to:
- `{patient}_transcripts_with_neural.csv`


In [1]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import fftconvolve
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
# -- Config -----------------------------------------------------------------
PATIENT = "YFA"
WRITE_ROOT = Path("/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new")
VAD_ROOT = WRITE_ROOT / PATIENT
VAD_DATA_DIR = VAD_ROOT / "vad_data"
TRANSCRIPTS_CSV = VAD_ROOT / f"{PATIENT}_transcripts.csv"

NEURAL_EMBEDDINGS_DIR = VAD_ROOT / "neural_embeddings"
NEURAL_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

COUNTS_PATH = NEURAL_EMBEDDINGS_DIR / "word_counts.npy"
COUNTS_OFFSETS_PATH = NEURAL_EMBEDDINGS_DIR / "word_spike_counts_offsets_all.npy"
FRS_PATH = NEURAL_EMBEDDINGS_DIR / "word_frs.npy"
DURS_PATH = NEURAL_EMBEDDINGS_DIR / "word_durs.npy"
MANIFEST_PATH = NEURAL_EMBEDDINGS_DIR / "word_neural_manifest.json"
TRANSCRIPTS_WITH_NEURAL_CSV = VAD_ROOT / f"{PATIENT}_transcripts_with_neural.csv"

FORCE_REBUILD = True
SPIKE_FS_EXPECTED = 1000
OFFSET_BINS = 0
SIGMA_MS = 50.0
TRUNCATE = 3.0
EDGE_MODE = "reflect"
ARRAY_DTYPE = np.float32

print("transcripts:", TRANSCRIPTS_CSV)
print("vad data:   ", VAD_DATA_DIR)
print("save dir:   ", NEURAL_EMBEDDINGS_DIR)


transcripts: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_transcripts.csv
vad data:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/vad_data
save dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/neural_embeddings


In [3]:
# -- Helpers ----------------------------------------------------------------
def gaussian_firing_rate(
    spike_bin,
    fs=1000,
    sigma_ms=50.0,
    truncate=3.0,
    edge_mode="reflect",
):
    """Return smoothed firing rates in Hz with shape (n_channels, n_bins)."""
    spike_bin = np.asarray(spike_bin, dtype=np.float32)
    n_ch, _ = spike_bin.shape

    sigma_bins = sigma_ms * fs / 1000.0
    radius = int(np.ceil(truncate * sigma_bins))
    radius = min(radius, max(spike_bin.shape[1] - 1, 0))
    if radius == 0:
        return spike_bin * fs
    t = np.arange(-radius, radius + 1, dtype=np.float32)
    kernel = np.exp(-0.5 * (t / sigma_bins) ** 2)
    kernel /= kernel.sum()

    def pad1d(x):
        if edge_mode == "reflect":
            left = x[:, 1:radius + 1][:, ::-1]
            right = x[:, -radius - 1:-1][:, ::-1]
        elif edge_mode == "nearest":
            left = np.repeat(x[:, :1], radius, axis=1)
            right = np.repeat(x[:, -1:], radius, axis=1)
        elif edge_mode == "constant":
            left = np.zeros((x.shape[0], radius), dtype=x.dtype)
            right = np.zeros((x.shape[0], radius), dtype=x.dtype)
        else:
            raise ValueError("edge_mode must be 'reflect', 'nearest', or 'constant'")
        return np.concatenate([left, x, right], axis=1)

    xpad = pad1d(spike_bin)
    rates = np.empty_like(spike_bin, dtype=np.float32)
    for c in range(n_ch):
        y = fftconvolve(xpad[c], kernel, mode="same")
        rates[c] = y[radius:-radius]

    return rates * fs


def get_spike_dir(interval_id: str) -> Path:
    return VAD_DATA_DIR / interval_id / "neural" / "spike_thresholding"


def get_spike_paths(interval_id: str):
    spike_dir = get_spike_dir(interval_id)
    spike_path = spike_dir / "binary_spiketrain.npy"
    stats_path = spike_dir / "spike_stats.json"
    success_path = spike_dir / "_SUCCESS"
    return spike_dir, spike_path, stats_path, success_path


def load_spike_interval(interval_id: str):
    spike_dir, spike_path, stats_path, success_path = get_spike_paths(interval_id)
    if not (success_path.exists() and spike_path.exists() and stats_path.exists()):
        raise FileNotFoundError(f"Missing spike-thresholding outputs for {interval_id}")

    spike_bin = np.load(spike_path, mmap_mode="r")
    with open(stats_path) as f:
        stats = json.load(f)
    return spike_bin, stats, spike_dir


def word_window_bins(row, fs: int, n_bins: int, offset_bins: int = 0):
    start_bin = int(np.floor(float(row.word_start_s) * fs)) + offset_bins
    end_bin = int(np.ceil(float(row.word_end_s) * fs)) + offset_bins

    start_bin = max(0, min(start_bin, max(n_bins - 1, 0)))
    end_bin = max(0, min(end_bin, n_bins))

    if n_bins == 0:
        return 0, 0

    # WhisperX words can become zero-width after discretization; keep at least one bin.
    if end_bin <= start_bin:
        end_bin = min(n_bins, start_bin + 1)

    return start_bin, end_bin


def ensure_overwrite_ok(paths, force_rebuild=False):
    existing = [p for p in paths if p.exists()]
    if existing and not force_rebuild:
        raise FileExistsError(
            "Outputs already exist. Set FORCE_REBUILD=True to overwrite:\n" +
            "\n".join(str(p) for p in existing)
        )


In [4]:
# -- Load transcripts + interval status -------------------------------------
transcripts = pd.read_csv(TRANSCRIPTS_CSV)
required_cols = ["interval_id", "word_start_s", "word_end_s"]
missing_cols = [c for c in required_cols if c not in transcripts.columns]
if missing_cols:
    raise ValueError(f"Transcript CSV is missing columns: {missing_cols}")

status_rows = []
for interval_id, group in transcripts.groupby("interval_id", sort=False):
    spike_dir, spike_path, stats_path, success_path = get_spike_paths(interval_id)
    row = {
        "interval_id": interval_id,
        "n_words": len(group),
        "spike_dir": spike_dir,
        "has_success": success_path.exists(),
        "has_spike_path": spike_path.exists(),
        "has_stats": stats_path.exists(),
    }
    if row["has_success"] and row["has_spike_path"] and row["has_stats"]:
        with open(stats_path) as f:
            stats = json.load(f)
        row["out_fs"] = int(stats.get("out_fs", -1))
        row["n_channels"] = int(stats.get("n_channels", -1))
        row["duration_s"] = float(stats.get("duration_s", np.nan))
    else:
        row["out_fs"] = np.nan
        row["n_channels"] = np.nan
        row["duration_s"] = np.nan
    row["ready"] = row["has_success"] and row["has_spike_path"] and row["has_stats"]
    status_rows.append(row)

status_df = pd.DataFrame(status_rows)
display(status_df.head(10))
print(f"total transcript rows: {len(transcripts)}")
print(f"intervals in transcript: {len(status_df)}")
print(f"intervals with spikes:   {int(status_df.ready.sum())}")
print(f"intervals missing data:  {int((~status_df.ready).sum())}")


,interval_id,n_words,spike_dir,has_success,has_spike_path,has_stats,out_fs,n_channels,duration_s,ready
0,20240423-124841_0004,1,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,545.014500,True
1,20240423-171352_0000,1350,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,599.600000,True
2,20240423-171352_0001,2351,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,1599.807433,True
3,20240423-171352_0002,161,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,122.075000,True
4,20240423-171352_0003,1371,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,839.014000,True
5,20240423-171352_0005,5,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,11.500000,True
6,20240423-171352_0006,1016,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,831.644000,True
7,20240423-171352_0007,412,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,259.854000,True
8,20240423-171352_0008,11,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,3.300000,True
9,20240423-171352_0009,273,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True,1000.0,40.0,227.593433,True


total transcript rows: 610928
intervals in transcript: 1030
intervals with spikes:   1025
intervals missing data:  5


In [5]:
# -- Preflight + output array creation --------------------------------------
ready_status = status_df[status_df["ready"]].copy()
if ready_status.empty:
    raise RuntimeError("No intervals are ready. Run spike thresholding first.")

unique_fs = sorted(ready_status["out_fs"].dropna().astype(int).unique().tolist())
unique_ch = sorted(ready_status["n_channels"].dropna().astype(int).unique().tolist())
print("out_fs values:", unique_fs)
print("n_channels values:", unique_ch)

if len(unique_fs) != 1:
    raise ValueError(f"Expected one shared spike sampling rate, got {unique_fs}")
if len(unique_ch) != 1:
    raise ValueError(f"Expected one shared channel count, got {unique_ch}")

OUT_FS = unique_fs[0]
N_CHANNELS = unique_ch[0]
if SPIKE_FS_EXPECTED is not None and OUT_FS != SPIKE_FS_EXPECTED:
    print(f"warning: expected {SPIKE_FS_EXPECTED} Hz but found {OUT_FS} Hz")

ensure_overwrite_ok(
    [COUNTS_PATH, COUNTS_OFFSETS_PATH, FRS_PATH, DURS_PATH, MANIFEST_PATH, TRANSCRIPTS_WITH_NEURAL_CSV],
    force_rebuild=FORCE_REBUILD,
)

n_words = len(transcripts)
counts_mm = np.lib.format.open_memmap(COUNTS_PATH, mode="w+", dtype=ARRAY_DTYPE, shape=(n_words, N_CHANNELS))
counts_offsets_mm = np.lib.format.open_memmap(COUNTS_OFFSETS_PATH, mode="w+", dtype=ARRAY_DTYPE, shape=(1, n_words, N_CHANNELS))
frs_mm = np.lib.format.open_memmap(FRS_PATH, mode="w+", dtype=ARRAY_DTYPE, shape=(n_words, N_CHANNELS))
durs_mm = np.lib.format.open_memmap(DURS_PATH, mode="w+", dtype=ARRAY_DTYPE, shape=(n_words,))

counts_mm[:] = np.nan
counts_offsets_mm[:] = np.nan
frs_mm[:] = np.nan
durs_mm[:] = np.nan

word_start_bins = np.full(n_words, -1, dtype=np.int32)
word_end_bins = np.full(n_words, -1, dtype=np.int32)
has_neural = np.zeros(n_words, dtype=bool)

print(f"allocated arrays for {n_words} words x {N_CHANNELS} channels")
print("counts path: ", COUNTS_PATH)
print("frs path:    ", FRS_PATH)


out_fs values: [1000]
n_channels values: [40]
allocated arrays for 610928 words x 40 channels
counts path:  /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/neural_embeddings/word_counts.npy
frs path:     /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/neural_embeddings/word_frs.npy


In [6]:
# -- Assemble counts / firing rates -----------------------------------------
missing_intervals = []
assembled_intervals = []

for interval_id, group in tqdm(transcripts.groupby("interval_id", sort=False), total=len(status_df)):
    row_idx = group.index.to_numpy()
    status_row = status_df.loc[status_df["interval_id"] == interval_id].iloc[0]

    if not bool(status_row["ready"]):
        missing_intervals.append(interval_id)
        continue

    spike_bin, stats, spike_dir = load_spike_interval(interval_id)
    fs = int(stats["out_fs"])
    n_bins = spike_bin.shape[1]
    gaussian_frs = gaussian_firing_rate(
        spike_bin,
        fs=fs,
        sigma_ms=SIGMA_MS,
        truncate=TRUNCATE,
        edge_mode=EDGE_MODE,
    ).astype(ARRAY_DTYPE, copy=False)

    local_counts = np.full((len(group), N_CHANNELS), np.nan, dtype=ARRAY_DTYPE)
    local_frs = np.full((len(group), N_CHANNELS), np.nan, dtype=ARRAY_DTYPE)
    local_durs = (group["word_end_s"].to_numpy() - group["word_start_s"].to_numpy()).astype(ARRAY_DTYPE)

    for j, row in enumerate(group.itertuples(index=False)):
        start_bin, end_bin = word_window_bins(row, fs=fs, n_bins=n_bins, offset_bins=OFFSET_BINS)
        if end_bin <= start_bin:
            continue

        local_counts[j] = spike_bin[:, start_bin:end_bin].sum(axis=1, dtype=np.uint32).astype(ARRAY_DTYPE)
        local_frs[j] = gaussian_frs[:, start_bin:end_bin].mean(axis=1).astype(ARRAY_DTYPE)
        word_start_bins[row_idx[j]] = start_bin
        word_end_bins[row_idx[j]] = end_bin
        has_neural[row_idx[j]] = True

    counts_mm[row_idx] = local_counts
    counts_offsets_mm[0, row_idx] = local_counts
    frs_mm[row_idx] = local_frs
    durs_mm[row_idx] = local_durs
    assembled_intervals.append(interval_id)

counts_mm.flush()
counts_offsets_mm.flush()
frs_mm.flush()
durs_mm.flush()

print(f"assembled intervals: {len(assembled_intervals)}")
print(f"missing intervals:   {len(missing_intervals)}")
if missing_intervals:
    print(missing_intervals[:10])


  0%|          | 0/1030 [00:00<?, ?it/s]

assembled intervals: 1025
missing intervals:   5
['20240424-141148_0021', '20240424-141148_0023', '20240424-141148_0024', '20240424-141148_0029', '20240426-115130_0163']


In [7]:
# -- Save transcript copy + manifest ----------------------------------------
transcripts_with_neural = transcripts.copy()
transcripts_with_neural["word_dur_s"] = np.asarray(durs_mm)
transcripts_with_neural["word_start_bin"] = word_start_bins
transcripts_with_neural["word_end_bin"] = word_end_bins
transcripts_with_neural["neural_row_idx"] = np.arange(len(transcripts_with_neural), dtype=np.int64)
transcripts_with_neural["has_neural_features"] = has_neural
transcripts_with_neural["word_counts_path"] = str(COUNTS_PATH)
transcripts_with_neural["word_frs_path"] = str(FRS_PATH)
transcripts_with_neural["word_durs_path"] = str(DURS_PATH)
transcripts_with_neural.to_csv(TRANSCRIPTS_WITH_NEURAL_CSV, index=False)

manifest = {
    "patient": PATIENT,
    "created_at": datetime.now().isoformat(),
    "transcripts_csv": str(TRANSCRIPTS_CSV),
    "transcripts_with_neural_csv": str(TRANSCRIPTS_WITH_NEURAL_CSV),
    "counts_path": str(COUNTS_PATH),
    "counts_offsets_path": str(COUNTS_OFFSETS_PATH),
    "frs_path": str(FRS_PATH),
    "durs_path": str(DURS_PATH),
    "n_words": int(len(transcripts_with_neural)),
    "n_channels": int(N_CHANNELS),
    "out_fs": int(OUT_FS),
    "offset_bins": int(OFFSET_BINS),
    "sigma_ms": float(SIGMA_MS),
    "truncate": float(TRUNCATE),
    "edge_mode": EDGE_MODE,
    "intervals_total": int(len(status_df)),
    "intervals_assembled": int(len(assembled_intervals)),
    "intervals_missing": missing_intervals,
    "valid_word_rows": int(has_neural.sum()),
    "array_dtype": str(np.dtype(ARRAY_DTYPE)),
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"saved transcript copy -> {TRANSCRIPTS_WITH_NEURAL_CSV}")
print(f"saved manifest       -> {MANIFEST_PATH}")
display(transcripts_with_neural.head(10))


saved transcript copy -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/YFA_transcripts_with_neural.csv
saved manifest       -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/neural_embeddings/word_neural_manifest.json


,patient,interval_id,toc,word,word_start_s,word_end_s,word_score,segment_idx,segment_text,segment_start_s,segment_end_s,speaker,interval_utc_start,interval_utc_end,interval_dur_s,br_ts,br_te,utc_word_start,utc_word_end,br_word_start,br_word_end,context,word_dur_s,word_start_bin,word_end_bin,neural_row_idx,has_neural_features,word_counts_path,word_frs_path,word_durs_path
0,YFA,20240423-124841_0004,20240423-124841,you,43.904,44.675,0.607,0,you,43.904,44.675,NaN,2024-04-23 20:24:41.962,2024-04-23 20:33:47.069,545.107,2175209868,2191560303,2024-04-23 20:25:25.866000000,2024-04-23 20:25:26.637000000,2176526988,2176550118,NaN,0.771,43904,44675,0,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YFA,20240423-171352_0000,20240423-171352,Leave,0.131,0.271,0.223,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.205000000,2024-04-23 22:13:53.345000000,2371743269,2371747469,NaN,0.140,131,271,1,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YFA,20240423-171352_0000,20240423-171352,everything,0.311,0.571,0.131,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.385000000,2024-04-23 22:13:53.645000000,2371748669,2371756469,Leave,0.260,311,571,2,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YFA,20240423-171352_0000,20240423-171352,behind,0.591,0.771,0.239,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.665000000,2024-04-23 22:13:53.845000000,2371757069,2371762469,Leave everything,0.180,591,771,3,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YFA,20240423-171352_0000,20240423-171352,that,0.791,0.891,0.708,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.865000000,2024-04-23 22:13:53.965000000,2371763069,2371766069,Leave everything behind,0.100,791,891,4,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YFA,20240423-171352_0000,20240423-171352,you,0.911,1.012,0.788,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:53.985000000,2024-04-23 22:13:54.086000000,2371766669,2371769699,Leave everything behind that,0.101,911,1012,5,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YFA,20240423-171352_0000,20240423-171352,don't,1.032,1.292,0.555,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:54.106000000,2024-04-23 22:13:54.366000000,2371770299,2371778099,Leave everything behind that you,0.260,1032,1292,6,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YFA,20240423-171352_0000,20240423-171352,need!,1.352,1.912,0.631,0,Leave everything behind that you don't need!,0.131,1.912,SPEAKER_01,2024-04-23 22:13:53.074,2024-04-23 22:23:52.674,599.600,2371739339,2389727339,2024-04-23 22:13:54.426000000,2024-04-23 22:13:54.98600

In [8]:
# -- Quick verification ------------------------------------------------------
counts = np.load(COUNTS_PATH, mmap_mode="r")
counts_offsets = np.load(COUNTS_OFFSETS_PATH, mmap_mode="r")
frs = np.load(FRS_PATH, mmap_mode="r")
durs = np.load(DURS_PATH, mmap_mode="r")
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print("counts shape:        ", counts.shape, counts.dtype)
print("counts offsets shape:", counts_offsets.shape, counts_offsets.dtype)
print("frs shape:           ", frs.shape, frs.dtype)
print("durs shape:          ", durs.shape, durs.dtype)
print("manifest valid rows: ", manifest["valid_word_rows"])

check_df = pd.DataFrame({
    "word": transcripts_with_neural["word"].head(10),
    "interval_id": transcripts_with_neural["interval_id"].head(10),
    "dur_s": durs[:10],
    "start_bin": transcripts_with_neural["word_start_bin"].head(10),
    "end_bin": transcripts_with_neural["word_end_bin"].head(10),
    "count_sum": np.nansum(counts[:10], axis=1),
    "fr_mean": np.nanmean(frs[:10], axis=1),
})
display(check_df)


counts shape:         (610928, 40) float32
counts offsets shape: (1, 610928, 40) float32
frs shape:            (610928, 40) float32
durs shape:           (610928,) float32
manifest valid rows:  594228


,word,interval_id,dur_s,start_bin,end_bin,count_sum,fr_mean
0,you,20240423-124841_0004,0.771,43904,44675,143.0,4.454332
1,Leave,20240423-171352_0000,0.140,131,271,103.0,18.104403
2,everything,20240423-171352_0000,0.260,311,571,301.0,27.287542
3,behind,20240423-171352_0000,0.180,591,771,226.0,29.015961
4,that,20240423-171352_0000,0.100,791,891,60.0,16.955921
5,you,20240423-171352_0000,0.101,911,1012,59.0,15.458376
6,don't,20240423-171352_0000,0.260,1032,1292,166.0,16.047626
7,need!,20240423-171352_0000,0.560,1352,1912,328.0,14.713312
8,Oh,20240423-171352_0000,0.240,2032,2272,181.0,18.462538
9,"no,",20240423-171352_0000,0.200,2312,2512,151.0,19.089108
